# Notebook 03 — Sequential Spatial-View Policy Analysis (`A0`–`A9`, V1 vs. CAL-Stop & Phase F)

**Purpose:** Examine the ten deterministic virtual-view operators (`A0`–`A9`), demonstrate canonical state construction, recursive arithmetic probability fusion, and stopping logic, compare `Learned Policy V1` against `Val-Ranked Fixed (CAL-Stop)` and the controlled five-spatial-action ablation (`Phase F`), and audit the documented `A7` Gabor saturation and `A9` non-semantic masking flaws.  
**Data & Artifact Requirement:** Aggregate trajectory curves, action selection frequencies, `Phase F` comparisons, and `A7`/`A9` forensic summaries execute standalone from tracked repository CSV/JSON ledgers. Visualizing `ViewGenerator` crops on a real dermoscopic image or auditing `policy.pt` / `per_action_test.npz` requires local ISIC imagery and frozen artifacts (`REQUIRES_DATA` / `REQUIRES_MODEL_ARTIFACT`).

## 1. Learning Objectives & Scientific Context

### Learning Objectives
1. Inspect all `A0`–`A9` deterministic operators and clarify that these are **virtual transformations of a single static photograph**, not physical camera acquisitions.
2. Contrast **recursive arithmetic running mean fusion** (`fuse_posterior`) against Bayesian posterior inference, and **supervised surrogate value estimation** (`MultiHeadActionPolicy`) against Bellman-optimized POMDPs.
3. Evaluate Risk/Accuracy vs. Mean Views across budgets `B = 0..5` for `Global A0`, `Learned Policy V1 (A1–A9)`, `Val-Ranked Fixed CAL-Stop`, `Val-Ranked Fixed Unstopped`, and `Phase F` restricted five-spatial-action masking (`{A1, A3, A4, A5, A6}`).
4. Quantify the documented `A7` Gabor 8-bit clipping/collapse flaw and `A9` non-semantic morphological masking behavior.

### Prerequisites
- Core packages (`torch`, `numpy`, `pandas`, `matplotlib`, `opencv-python`).

## 2. Environment & Artifact Preflight

In [ ]:
from __future__ import annotations

import json
import os
import platform
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt


def find_repo_root(start: Path | None = None) -> Path:
    """Locate repository root portably without hardcoded machine paths."""
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "research" / "derma_act" / "config.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate DERMA-ACT repository root containing pyproject.toml and research/derma_act/config.py")


def find_tables_dir(repo_root: Path) -> Path | None:
    for rel in [
        Path("supplementary") / "tables",
        Path("manuscripts") / "current" / "PAPER_DRAFT_Q2_V3_2_1_PRE_SUBMISSION" / "tables",
    ]:
        cand = repo_root / rel
        if cand.is_dir():
            return cand
    return None


SEED = 42
random.seed(SEED)
np.random.seed(SEED)

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import hashlib
from research.derma_act.config import ACTION_DISPLAY, POLICY_PATH, EXPECTED_HASHES, load_lambda, load_tau
from src.modules.active_perception.action_space import ObservationAction
from src.modules.active_perception.actions import action_index_to_token, candidate_index_to_token
from src.modules.active_perception.view_generator import ViewGenerator

OUT_DIR = REPO_ROOT / "outputs" / "notebooks" / "nb03_sequential_policy"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIG4_CSV = REPO_ROOT / "data" / "aggregate" / "FIGURE_4_V1_1_DATA.csv"
REG_CSV = REPO_ROOT / "research" / "derma_act" / "p5_6_action_quality_audit" / "P5_6_ACTION_QUALITY_REGISTRY.csv"
FREQ_CSV = REPO_ROOT / "research" / "derma_act" / "p5_6_action_quality_audit" / "P5_6_POLICY_ACTION_FREQUENCIES.csv"
A7_JSON = REPO_ROOT / "research" / "derma_act" / "p5_6_action_quality_audit" / "P5_6_A7_COHORT_SUMMARY.json"
A9_JSON = REPO_ROOT / "research" / "derma_act" / "p5_6_action_quality_audit" / "P5_6_A9_COHORT_SUMMARY.json"
PHASE_F_CSV = REPO_ROOT / "research" / "derma_act" / "p8_1_final_submission_audit" / "phase_f_restricted_policy" / "phase_f_comparison_table.csv"
PHASE_F_TESTS = REPO_ROOT / "research" / "derma_act" / "p8_1_final_submission_audit" / "phase_f_restricted_policy" / "phase_f_paired_tests.csv"

IMG_DIRS = [
    REPO_ROOT / "data" / "isic2019" / "ISIC_2019_Training_Input",
    REPO_ROOT / "data" / "ISIC_2019_Training_Input",
]
SAMPLE_IMG = None
for d in IMG_DIRS:
    if d.is_dir():
        pref = d / "ISIC_0012970.jpg"
        if pref.is_file():
            SAMPLE_IMG = pref
            break
        first_jpg = next(iter(sorted(d.glob("*.jpg"))), None)
        if first_jpg is not None:
            SAMPLE_IMG = first_jpg
            break
HAS_SAMPLE_IMG = SAMPLE_IMG is not None and SAMPLE_IMG.is_file()
PER_ACT_TEST = REPO_ROOT / "research" / "derma_act" / "runs" / "derma_act_v0" / "predictions" / "per_action_test.npz"

display(pd.DataFrame([
    {"artifact": "FIGURE_4_V1_1_DATA.csv", "available": FIG4_CSV.is_file(), "tier": "TRACKED"},
    {"artifact": "P5_6_ACTION_QUALITY_REGISTRY.csv", "available": REG_CSV.is_file(), "tier": "TRACKED"},
    {"artifact": "P5_6_A7/A9 cohort summaries", "available": A7_JSON.is_file() and A9_JSON.is_file(), "tier": "TRACKED"},
    {"artifact": "phase_f_comparison_table.csv", "available": PHASE_F_CSV.is_file(), "tier": "TRACKED"},
    {"artifact": "Sample ISIC JPEG (local)", "available": HAS_SAMPLE_IMG, "tier": "REQUIRES_DATA"},
    {"artifact": "policy.pt & per_action_test.npz", "available": POLICY_PATH.is_file() and PER_ACT_TEST.is_file(), "tier": "REQUIRES_MODEL_ARTIFACT"},
]))

## 3. Action Space Inventory (`A0`–`A9`), Operator Audits & Fusion Mechanics

1. **Deterministic Virtual Views (`ViewGenerator`):** Every candidate action `A1`–`A9` transforms the **same** input dermoscopic image into a `224x224` RGB view; no new physical sensor measurement is taken.
2. **Documented Operator Caveats:**
   - **`A7` (Gabor Texture):** `cv2.filter2D(gray, cv2.CV_8UC3, kernel)` clips positive filter responses into unsigned 8-bit integers prior to max-pooling, yielding `99.97%` median saturation and `25.69%` (`910/3,542`) all-zero normalized collapse on Validation.
   - **`A8` (Shades-of-Gray Color Constancy, $p=6$):** Implemented in `ObservationAction.COLOR_NORMALIZED` (legacy token label `'Color Suppressed'` in `config.py`).
   - **`A9` (Morphological Black-Hat + Telea Inpainting):** Thresholds a `17x17` black-hat transform at `> 10`, masking `11.55%` of pixels on average across dark structures indiscriminately — **not** a semantic hair detector.
3. **Fusion & Policy Rules:** Sequential predictions combine via unweighted recursive arithmetic averaging $p_{t+1} = (b p_t + p_{\mathrm{cand}}) / (b + 1)$ (**not** Bayesian updating). `MultiHeadActionPolicy` is a finite-horizon supervised surrogate trained on one-step targets (**not** a Bellman-optimized POMDP).

In [ ]:
reg_df = pd.read_csv(REG_CSV)
freq_df = pd.read_csv(FREQ_CSV)
a7_sum = json.loads(A7_JSON.read_text(encoding="utf-8"))
a9_sum = json.loads(A9_JSON.read_text(encoding="utf-8"))

cols_show = ["action_token", "display_name", "action_category", "implementation_integrity", "test_selection_share_pct", "test_cohort_exposure_pct", "standalone_accuracy", "fixed_comparator_status"]
print("=== Canonical Action Quality & Policy Utilization Registry ===")
display(reg_df[cols_show])

print(
    f"A7 Val Audit (N={a7_sum['processed_count']}): median saturation = {100*a7_sum['effective_saturation_fraction']['median']:.2f}%, "
    f"all-zero collapse = {a7_sum['threshold_proportions']['all_zero_normalized_collapse_count']}/{a7_sum['processed_count']} "
    f"({100*a7_sum['threshold_proportions']['all_zero_normalized_collapse_proportion']:.2f}%)"
)
print(
    f"A9 Val Audit (N={a9_sum['processed_count']}): mean masked area = {100*a9_sum['mask_coverage_fraction']['mean']:.2f}% "
    f"(median {100*a9_sum['mask_coverage_fraction']['median']:.2f}%, p95 {100*a9_sum['mask_coverage_fraction']['p95']:.2f}%)"
)

## 4. Benchmark Comparison: Learned V1 vs. Fixed CAL-Stop vs. Restricted 5-Spatial-Action Ablation (`Phase F`)

In [ ]:
fig4_df = pd.read_csv(FIG4_CSV)
pf_df = pd.read_csv(PHASE_F_CSV)
pf_tests = pd.read_csv(PHASE_F_TESTS)

# Summarize B=5 strategies cleanly without collapsing different action inventories
b5_main = fig4_df[fig4_df["budget"] == 5][["strategy", "mean_cost", "risk", "accuracy", "balanced_accuracy", "stop_freq", "holm_adjusted_p_value"]].copy()
b5_pf = pf_df[pf_df["budget"] == 5].iloc[0]
print("=== Held-Out Test B=5 Comparison (N=3,686, K=1,972) ===")
display(b5_main)
print(
    f"Phase F Restricted Spatial V1 (B=5, {{A1,A3,A4,A5,A6}}): mean_views={b5_pf['mean_cost']:.2f}, "
    f"risk={100*b5_pf['risk']:.2f}%, acc={100*b5_pf['accuracy']:.2f}%, bal_acc={100*b5_pf['balanced_accuracy']:.2f}%, "
    f"early_stop={100*b5_pf['early_stop_freq']:.2f}%"
)
display(pf_tests[pf_tests["budget"] == 5][["comparison", "risk_diff", "mcnemar_p", "holm_p_adj_m20"]])

# Visualization: Risk & Accuracy vs Mean Views + Realized Action Composition
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))

a0_pt = fig4_df[fig4_df["strategy"] == "Global_A0"].iloc[0]
axes[0].errorbar([0.0], [100 * a0_pt["risk"]], yerr=[[100 * (a0_pt["risk"] - a0_pt["risk_ci95_lower"])], [100 * (a0_pt["risk_ci95_upper"] - a0_pt["risk"])]], fmt="ko", capsize=4, label="Global A0 (B=0)")

plot_specs = [
    ("Learned_V1", "Learned V1 (9-Action A1-A9)", "#2b6cb0", "o-"),
    ("Val_Ranked_ConfStop_Independent", "Val-Ranked Fixed (CAL-Stop, 5 Spatial)", "#c53030", "s--"),
    ("Val_Ranked_Unstopped", "Val-Ranked Fixed (Unstopped, 5 Spatial)", "#718096", "^:"),
]
for key, label, col, style in plot_specs:
    sub = fig4_df[fig4_df["strategy"] == key].sort_values("budget")
    if not sub.empty:
        axes[0].plot(sub["mean_cost"], 100 * sub["risk"], style, color=col, linewidth=2, label=label)

axes[0].plot(pf_df["mean_cost"], 100 * pf_df["risk"], "D-", color="#2f855a", linewidth=2, label="Restricted Spatial V1 (Phase F, 5 Spatial)")
axes[0].set_xlabel("Mean Post-A0 Virtual Views Acquired")
axes[0].set_ylabel("Held-Out Test Top-1 Error / Risk (%)")
axes[0].set_title("Risk vs. Acquisition Cost (Budgets B = 0..5, N = 3,686)")
axes[0].legend(fontsize=8, loc="upper right")
axes[0].grid(alpha=0.3)

# Panel B: Step-by-step action selection counts for Learned V1 (A1-A9)
cand_freq = freq_df[freq_df["action_token"] != "A0"].copy()
axes[1].bar(cand_freq["action_token"], cand_freq["share_of_virtual_actions_pct"], color=["#2b6cb0" if s == "Included" else "#dd6b20" for s in cand_freq["fixed_comparator_status"]])
axes[1].set_xlabel("Candidate Virtual Action (Blue = in Fixed Spatial Set; Orange = V1-Only)")
axes[1].set_ylabel("Share of 14,815 V1 Post-A0 Selections (%)")
axes[1].set_title("Learned Policy V1 Action Utilization at B = 5")
axes[1].grid(axis="y", alpha=0.3)
for i, v in enumerate(cand_freq["share_of_virtual_actions_pct"]):
    axes[1].text(i, v + 0.4, f"{v:.1f}%", ha="center", fontsize=8)

plt.tight_layout()
fig.savefig(OUT_DIR / "sequential_policy_analysis.png", dpi=150)
plt.show()

## 5. Optional Local Visual Inspection of `ViewGenerator` & Frozen Policy Check (`REQUIRES_DATA` / `REQUIRES_MODEL_ARTIFACT`)

If a local ISIC 2019 sample image (`data/ISIC_2019_Training_Input/ISIC_0012970.jpg`) is available, this cell renders its deterministic `A0`–`A9` views directly using `ViewGenerator` (and inspects `policy.pt` SHA-256 & `per_action_test.npz` shape when present). Otherwise, it skips cleanly without synthetic placeholder images.

In [ ]:
import cv2

if POLICY_PATH.is_file():
    pol_sha = hashlib.sha256(POLICY_PATH.read_bytes()).hexdigest()
    print(f"policy.pt SHA-256 verified match: {pol_sha == EXPECTED_HASHES['policy']} ({pol_sha[:16]}...)")
else:
    print("SKIPPED (REQUIRES_MODEL_ARTIFACT): policy.pt not present in this clean clone.")

if PER_ACT_TEST.is_file():
    pa = np.load(PER_ACT_TEST, allow_pickle=True)
    print("per_action_test.npz keys:", {k: pa[k].shape for k in pa.files})
else:
    print("SKIPPED (REQUIRES_MODEL_ARTIFACT): per_action_test.npz not present in this clean clone.")

if HAS_SAMPLE_IMG:
    bgr = cv2.imread(str(SAMPLE_IMG))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    vg = ViewGenerator(image_size=224)
    action_map = [
        ("A0", ObservationAction.KEEP_FULL),
        ("A1", ObservationAction.ZOOM_CENTER),
        ("A2", ObservationAction.ZOOM_BORDER),
        ("A3", ObservationAction.TOP_REGION),
        ("A4", ObservationAction.BOTTOM_REGION),
        ("A5", ObservationAction.LEFT_REGION),
        ("A6", ObservationAction.RIGHT_REGION),
        ("A7", ObservationAction.TEXTURE_REGION),
        ("A8", ObservationAction.COLOR_NORMALIZED),
        ("A9", ObservationAction.ARTIFACT_SUPPRESSED),
    ]
    fig, axes = plt.subplots(2, 5, figsize=(12, 5.0))
    for ax, (tok, act) in zip(axes.flat, action_map):
        view = vg.generate(rgb, act)
        ax.imshow(view)
        ax.set_title(f"{tok}: {ACTION_DISPLAY[tok]}", fontsize=8)
        ax.axis("off")
    plt.suptitle("Deterministic Virtual Views A0–A9 Generated via ViewGenerator (Local Sample)", fontsize=10)
    plt.tight_layout()
    fig.savefig(OUT_DIR / "local_sample_views_A0_A9.png", dpi=130)
    plt.show()
else:
    print("SKIPPED (REQUIRES_DATA): Local ISIC JPEG not present; virtual-view rendering skipped cleanly.")

## 6. Interpretation & Scientific Boundaries

- **No Pareto Dominance / No Claim of Equivalence:** At `B = 5`, full Learned V1 (`26.94%` error, `4.02` views) does not beat `Val-Ranked Fixed CAL-Stop` (`25.96%` error, `3.90` views, Holm $p_{\mathrm{adj}} = 0.3185$). Under equalized five-spatial-action masking (`Phase F`), Spatial V1 achieves `26.21%` error at `3.44` mean views (`p_adj = 1.000` vs `CAL-Stop`), which does **not** prove equivalence or substitute for retraining a 5-action policy from scratch.
- **Class Trade-Offs:** Full V1 increases sensitivity on `AKIEC` (`+9.05 pp`, `p_adj = 0.0016`) relative to `CAL-Stop`, while reducing sensitivity on `NV` (`-3.10 pp`, `p_adj = 6.80e-11`) and `BKL` (`-5.78 pp`, `p_adj = 0.0011`).
- **Operator Limitations:** `A7` rarity (`6 / 14,815` selections) cannot be interpreted as proof of learned defect rejection, and `A9` is non-semantic morphological masking rather than clinical hair detection.

### Reproduction & References
- Output saved to: `outputs/notebooks/nb03_sequential_policy/sequential_policy_analysis.png`
- Related scripts/docs: [`scripts/run_phase_f_restricted_policy.py`](../scripts/run_phase_f_restricted_policy.py) · [Architecture](../docs/ARCHITECTURE.md) · [Limitations](../docs/LIMITATIONS.md)